In [1]:
# Set up the pinned official V-JEPA source and check the GPU.
import sys, subprocess
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.26', 'einops', 'pillow'], check=True)
upstream = Path('/content/vjepa2')
if not upstream.exists():
    subprocess.run(['git', 'clone', 'https://github.com/facebookresearch/vjepa2.git', str(upstream)], check=True)
subprocess.run(['git', '-C', str(upstream), 'checkout', '204698b45b3712590f06245fbfba32d3be539812'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
GPU: Tesla T4
GPU memory GiB: 14.56


In [2]:
# Load frozen pretrained ViT-L and verify one clip fits.
import gc, time
sys.path.insert(0, str(upstream))
from src.hub import backbones
backbones.VJEPA_BASE_URL = 'https://dl.fbaipublicfiles.com/vjepa2'
encoder, predictor = backbones.vjepa2_1_vit_large_384(pretrained=True)
del predictor
gc.collect()
encoder = encoder.eval().requires_grad_(False).to('cuda')
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
    smoke = encoder(torch.zeros(1, 3, 16, 384, 384, device='cuda'))
assert tuple(smoke.shape) == (1, 4608, 1024), smoke.shape
assert torch.isfinite(smoke).all()
print('Encoder output:', tuple(smoke.shape))
print('Peak GPU GiB:', round(torch.cuda.max_memory_allocated()/1024**3, 3))
print('Smoke seconds:', round(time.perf_counter()-t0, 2))
del smoke


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Downloading: "https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitl_dist_vitG_384.pt" to /root/.cache/torch/hub/checkpoints/vjepa2_1_vitl_dist_vitG_384.pt


100%|██████████| 4.80G/4.80G [00:37<00:00, 138MB/s]
/usr/lib/python3.13/contextlib.py:109: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Encoder output: (1, 4608, 1024)
Peak GPU GiB: 1.339
Smoke seconds: 2.9


In [3]:
# Run the pinned experiment. Ground truth after initialization is for scoring only.
import contextlib, json
project = Path('/content/vjepa-poc')
if not project.exists():
    subprocess.run(['git', 'clone', '--branch', 'experiment/part-consistency-colab', 'https://github.com/skbdps/vjepa-poc.git', str(project)], check=True)
subprocess.run(['git', '-C', str(project), 'checkout', '461dd0264dfe023bc9639479a388fa2eda785692'], check=True)
sys.path.insert(0, str(project / 'experiments/day3'))
import part_consistency, run_experiment
part_consistency.self_test('/content/day3_self_test')
run_dir = Path('/content/day3_run')
run_dir.mkdir(exist_ok=True)
print('Running 3 calibration and 6 held-out clips, two encoder windows each...')
with (run_dir / 'run.log').open('w') as log, contextlib.redirect_stdout(log):
    report = run_experiment.run(run_dir, upstream, encoder=encoder)
print('EXPERIMENT_COMPLETE')
print(json.dumps({k: v['overall'] for k, v in report['methods'].items()}, indent=2))
print((run_dir / 'manifest.json').read_text())


{
  "status": "passed",
  "oracle_fixture_accuracy": 1.0,
  "identity_swap_detected": true,
  "future_labels_not_matcher_input": true,
  "real_model_results": false,
  "visible_denominator": 315,
  "absent_denominator": 36,
  "recovery_denominator": 4
}
Running 3 calibration and 6 held-out clips, two encoder windows each...


/usr/lib/python3.13/contextlib.py:109: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


EXPERIMENT_COMPLETE
{
  "vjepa": {
    "localization_accuracy_given_visible": {
      "numerator": 276,
      "denominator": 315,
      "rate": 0.8761904761904762
    },
    "wrong_car_given_visible": {
      "numerator": 1,
      "denominator": 315,
      "rate": 0.0031746031746031746
    },
    "wrong_part_given_visible": {
      "numerator": 0,
      "denominator": 315,
      "rate": 0.0
    },
    "presence_recall_given_visible": {
      "numerator": 294,
      "denominator": 315,
      "rate": 0.9333333333333333
    },
    "false_presence_given_absent": {
      "numerator": 6,
      "denominator": 36,
      "rate": 0.16666666666666666
    },
    "recovery_accuracy": {
      "numerator": 2,
      "denominator": 4,
      "rate": 0.5
    },
    "ambiguous_samples_skipped": 9,
    "uniform_patch_chance_given_visible": 0.010146604938271605
  },
  "rgb_patch_mean": {
    "localization_accuracy_given_visible": {
      "numerator": 11,
      "denominator": 315,
      "rate": 0.03492063492

In [4]:
# Results: localization is measured at patch/tubelet resolution.
import pandas as pd
from IPython.display import display, HTML, Video
rows = []
for method, values in report['methods'].items():
    m = values['overall']
    def ratio(metric):
        z = m[metric]
        return f"{z['numerator']}/{z['denominator']}"
    rows.append({'Method': method, 'Correct visible': ratio('localization_accuracy_given_visible'), 'Accuracy %': round(100*m['localization_accuracy_given_visible']['rate'],1), 'False presence when hidden': ratio('false_presence_given_absent'), 'Recovered after occlusion': ratio('recovery_accuracy'), 'Second-window accuracy %': round(100*values['by_window']['across_window']['localization_accuracy_given_visible']['rate'],1)})
display(HTML('<h2>Day 3: part consistency results</h2><p>Frozen V-JEPA 2.1 ViT-L on a Tesla T4. 3 calibration clips, 6 held-out synthetic clips. Four tagged parts per clip; 315 visible checks, 36 absent checks, 4 recovery events. Nine ambiguous checks excluded.</p>'))
display(pd.DataFrame(rows).set_index('Method'))
display(HTML('<p><b>Interpretation:</b> V-JEPA ties optical flow on visible localization in this small diagnostic, with fewer false-presence errors and 2/4 recoveries. Cars stay in separate lanes. This does not establish camera-angle robustness, face consistency or generative editing. Both-frame annotations initialize V-JEPA; optical flow receives frame 0. Features are offline within each 16-frame window.</p>'))
for video in sorted((run_dir / 'results').glob('*comparison.mp4')):
    print(video.name)
    display(Video(str(video), embed=True, width=640))


,Correct visible,Accuracy %,False presence when hidden,Recovered after occlusion,Second-window accuracy %
Method,,,,,
vjepa,276/315,87.6,6/36,2/4,88.0
rgb_patch_mean,11/315,3.5,6/36,0/4,3.0
fixed_initial_position,41/315,13.0,36/36,0/4,2.4
lucas_kanade_flow,276/315,87.6,36/36,0/4,77.2


evaluation_200_comparison.mp4


evaluation_201_comparison.mp4


In [5]:
# Preserve all results and reusable feature caches, excluding model weights.
import shutil
from google.colab import files
manifest_path = run_dir / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
manifest['experiment_commit'] = '461dd0264dfe023bc9639479a388fa2eda785692'
manifest['run_date_IST'] = '2026-10-09'
manifest_path.write_text(json.dumps(manifest, indent=2))
shutil.copy2('/content/day3_self_test/self_test.json', run_dir / 'evaluator_self_test.json')
archive = shutil.make_archive('/content/VJEPA_Day3_Run_2026-10-09', 'zip', run_dir)
print('Completed archive:', Path(archive).name, round(Path(archive).stat().st_size/1024**2, 1), 'MiB')
files.download(archive)


Completed archive: VJEPA_Day3_Run_2026-10-09.zip 150.8 MiB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>